**1)**
Set up the parameters

In [1]:
# 之前提交的作业DDM少画了一张图，就是能直观显示DDM变种的决策过程的模拟图，这是修改版，补上了这一点


# DDM实验中，我补充实现了collision bound和leaky 两种变形，均取得不错结果。
# 其中bound使用的一个线性下降的模型，当然也可以用高斯或者二次等等，得到了提前收敛的结果，但是准确率略微下降，这是迫于时间压力的结果。
# leaky能够延迟达到边界的时间，也符合预期，准确率略微下降，因为扰动和噪声相当于增加了，影响力证据积累。

using Distributions
using PlotlyJS
using Random

# --- DDM Parameters ---
k = 0.3  # Drift rate (evidence accumulation rate)
σ = 1.0   # Noise standard deviation
B = 1.5   # Decision boundary
B_max = B   # collapsing_upper_bound
B_min = 1.0   # collapsing_lower_bound
tau = 2.0
λ = 0.005  # Leak coefficient
dt = 0.001 # Time step
max_t = 5.0 # Maximum simulation time
max_t_steps = Int(max_t/dt)


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24


5000

**2)**
Here is the main function for simulating DDM

In [2]:
# --- Simulation Function ---
function simulate_ddm(k, σ, B, dt, max_t)
    x = 0.0  # Initial decision variable
    xs = zeros(max_t_steps)
    r = 0    # undetermined
    t_step = 1

    while abs(x) < B && t_step < max_t_steps
        dx = k * dt + σ * randn() * sqrt(dt) # Euler-Maruyama integration
        x += dx
        xs[t_step] = x
        t_step = t_step + 1
    end
    if x>=B
        r=1
        xs[t_step:max_t_steps] .= B
    elseif x<=-B
        r=-1
        xs[t_step:max_t_steps] .= -B
    end
    return t_step * dt, xs, r # 1 for upper boundary, -1 for lower
end


simulate_ddm (generic function with 1 method)

**homework**
Here is the main function for simulating DDM with collapsing bound

In [3]:
# --- Simulation Function ---
function simulate_ddm_with_collapsing_bound(k, σ, B_max, B_min, tau, dt, max_t)
    x = 0.0  # Initial decision variable
    xs = zeros(max_t_steps)
    Bs = zeros(max_t_steps)  # Array to store boundary changes
    r = 0    # undetermined
    t_step = 1
    B = B_max
    while abs(x) < B && t_step < max_t_steps
        B = B_min + (B_max - B_min) * exp(- (t_step * dt) / tau) # Update the boundary
        dx = k * dt + σ * randn() * sqrt(dt) # Euler-Maruyama integration
        x += dx
        xs[t_step] = x
        Bs[t_step] = B
        t_step = t_step + 1
    end
    if x>=B
        r=1
        xs[t_step:max_t_steps] .= B
        Bs[t_step:max_t_steps] .= B
    elseif x<=-B
        r=-1
        xs[t_step:max_t_steps] .= -B
        Bs[t_step:max_t_steps] .= B
    end
    return t_step * dt, xs, r, Bs # 1 for upper boundary, -1 for lower
end


simulate_ddm_with_collapsing_bound (generic function with 1 method)

**homework**
Here is the main function for simulating DDM with leaky integration

In [4]:
# --- Simulation Function --- 由于此时直接泄露可能会导致x一直很小所以决定达到阈值开始泄露
function simulate_ddm_leaky_integration(k, σ, B, λ, dt, max_t)
    x = 0.0  # Initial decision variable
    xs = zeros(max_t_steps)
    r = 0    # undetermined
    t_step = 1

    while abs(x) < B && t_step < max_t_steps
        dx = k * dt + σ * randn() * sqrt(dt) # Euler-Maruyama integration
        if abs(x) >= 0.9 * B
            x = dx + x * (1 - λ)  # Apply leak
        else
            x += dx  # No leak applied
        end
        xs[t_step] = x
        t_step = t_step + 1
    end
    if x>=B
        r=1
        xs[t_step:max_t_steps] .= B
    elseif x<=-B
        r=-1
        xs[t_step:max_t_steps] .= -B
    end
    return t_step * dt, xs, r # 1 for upper boundary, -1 for lower
end


simulate_ddm_leaky_integration (generic function with 1 method)

**3)**
Let's run a bunch of simulations, recording the reaction times and the average drifting trajectory.

In [5]:
# --- Run Multiple Trials ---
n_trials = 10000
decisions = zeros(Int, n_trials)
rts_correct = []
rts_error = []
n_correct = n_error = 0
trace_correct = zeros(max_t_steps)
trace_error = zeros(max_t_steps)

for i in 1:n_trials
    rt, xs, decision = simulate_ddm(k, σ, B, dt, max_t)
    decisions[i] = decision
    # Record reaction time
    if decision == 1 
        push!(rts_correct, rt) 
        n_correct = n_correct + 1
        trace_correct = ((n_correct-1) * trace_correct + xs) / n_correct
    elseif decision == -1
        push!(rts_error, rt)
        n_error = n_error + 1
        trace_error = ((n_error-1) * trace_error + xs) / n_error
    end
end


**作业部分**

In [6]:
# DDM 模型建立部分

# collapsing bound
decisions_collapse = zeros(Int, n_trials)
rts_correct_collapse = []
rts_error_collapse = []
n_correct_collapse = n_error_collapse = 0
trace_correct_collapse = zeros(max_t_steps)
trace_error_collapse = zeros(max_t_steps)
trace_bound_correct_collapse = zeros(max_t_steps)
trace_bound_error_collapse = zeros(max_t_steps)

for trial in 1:n_trials
    rt_collapse, xs_collapse, decision_collapse, bs_collapse = simulate_ddm_with_collapsing_bound(k, σ, B_max, B_min, tau, dt, max_t)
    decisions_collapse[trial] = decision_collapse
    # 记录反应时间
    if decision_collapse == 1 
        push!(rts_correct_collapse, rt_collapse) 
        n_correct_collapse += 1
        trace_correct_collapse = ((n_correct_collapse - 1) * trace_correct_collapse + xs_collapse) / n_correct_collapse
        trace_bound_correct_collapse = ((n_correct_collapse - 1) * trace_bound_correct_collapse + bs_collapse) / n_correct_collapse
    elseif decision_collapse == -1
        push!(rts_error_collapse, rt_collapse)
        n_error_collapse += 1
        trace_error_collapse = ((n_error_collapse - 1) * trace_error_collapse + xs_collapse) / n_error_collapse
        trace_bound_error_collapse = ((n_error_collapse - 1) * trace_bound_error_collapse + bs_collapse) / n_error_collapse
    end
end


# leaky 
decisions_leak = zeros(Int, n_trials)
rts_correct_leak = []
rts_error_leak = []
n_correct_leak = n_error_leak = 0
trace_correct_leak = zeros(max_t_steps)
trace_error_leak = zeros(max_t_steps)

for trial in 1:n_trials 
    rt_leak, xs_leak, decision_leak = simulate_ddm_leaky_integration(k, σ, B, λ, dt, max_t)
    decisions_leak[trial] = decision_leak
    # 记录反应时间
    if decision_leak == 1 
        push!(rts_correct_leak, rt_leak) 
        n_correct_leak += 1
        trace_correct_leak = ((n_correct_leak - 1) * trace_correct_leak + xs_leak) / n_correct_leak
    elseif decision_leak == -1
        push!(rts_error_leak, rt_leak)
        n_error_leak += 1
        trace_error_leak = ((n_error_leak - 1) * trace_error_leak + xs_leak) / n_error_leak
    end
end

In [7]:
# DDM 模型拟合作图

# Collapsing bound
histogram_correct_collapse = histogram(x=rts_correct_collapse, nbinsx=20, name="Correct", opacity=0.6)
histogram_error_collapse = histogram(x=rts_error_collapse, nbinsx=20, name="Error", opacity=0.6)

layout_hist_collapse = Layout(title="Reaction Time Distribution Collapsing", xaxis_title="Time (s)", yaxis_title="Frequency", bar_mode="overlay")
display(plot([histogram_correct_collapse, histogram_error_collapse], layout_hist_collapse))

# 平均漂移轨迹
trajectory_correct_collapse = scatter(x=1:max_t_steps, y=trace_correct_collapse, mode="lines", name="Correct Trials")
trajectory_error_collapse = scatter(x=1:max_t_steps, y=trace_error_collapse, mode="lines", name="Error Trials")
upper_bound_collapse = scatter(x=1:max_t_steps, y=trace_bound_correct_collapse, mode="lines", name="Upper Bound", line=attr(dash="dash"))
lower_bound_collapse = scatter(x=1:max_t_steps, y=-trace_bound_error_collapse, mode="lines", name="Lower Bound", line=attr(dash="dash"))

layout_traj_collapse = Layout(title="Average DDM Trajectory Collapsing", xaxis_title="Time (s)", yaxis_title="x")

display(plot([trajectory_correct_collapse, trajectory_error_collapse, upper_bound_collapse, lower_bound_collapse], layout_traj_collapse))

# 分析
accuracy_collapse = n_correct_collapse / (n_correct_collapse + n_error_collapse) # 假设 '1' 是正确选择

println("Accuracy: ", accuracy_collapse)
println("Mean RT (correct): ", mean(rts_correct_collapse))
println("Mean RT (error): ", mean(rts_error_collapse))


# leaky
# 反应时间直方图
histogram_correct_leak = histogram(x=rts_correct_leak, nbinsx=20, name="Correct", opacity=0.6)
histogram_error_leak = histogram(x=rts_error_leak, nbinsx=20, name="Error", opacity=0.6)

layout_hist_leak = Layout(title="Reaction Time Distribution (Leaky)", xaxis_title="Time (s)", yaxis_title="Frequency", bar_mode="overlay")
display(plot([histogram_correct_leak, histogram_error_leak], layout_hist_leak))

# 平均漂移轨迹
trajectory_correct_leak = scatter(x=1:max_t_steps, y=trace_correct_leak, mode="lines", name="Correct Trials")
trajectory_error_leak = scatter(x=1:max_t_steps, y=trace_error_leak, mode="lines", name="Error Trials")
upper_bound_leak = scatter(x=[1, max_t_steps], y=[B,B], mode="lines", name="Upper Bound", line=attr(dash="dash"))
lower_bound_leak = scatter(x=[1, max_t_steps], y=[-B,-B], mode="lines", name="Lower Bound", line=attr(dash="dash"))

layout_traj_leak = Layout(title="Average DDM Trajectory (Leaky)", xaxis_title="Time (s)", yaxis_title="x")

display(plot([trajectory_correct_leak, trajectory_error_leak, upper_bound_leak, lower_bound_leak], layout_traj_leak))

# 分析
accuracy_leak = n_correct_leak / (n_correct_leak + n_error_leak) # 假设 '1' 是正确选择

println("Accuracy (Leaky): ", accuracy_leak)
println("Mean RT (correct, Leaky): ", mean(rts_correct_leak))
println("Mean RT (error, Leaky): ", mean(rts_error_leak))

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "histogram with fields name, nbinsx, opacity, type, and x",
  "histogram with fields name, nbinsx, opacity, type, and x"
]

layout: "layout with fields bar, margin, template, title, xaxis, and yaxis"

data: [
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

Accuracy: 

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24


...

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "histogram with fields name, nbinsx, opacity, type, and x",
  "histogram with fields name, nbinsx, opacity, type, and x"
]

layout: "layout with fields bar, margin, template, title, xaxis, and yaxis"

data: [
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y",
  "scatter with fields line, mode, name, type, x, and y"
]

layout: "layout with fields margin, template, title, xaxis, and yaxis"

Accuracy (Leaky): 0.7538383453876412
Mean RT (correct, Leaky): 2.180870223868036
Mean RT (error, Leaky): 2.1417912371134014


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24


In [9]:
# DDM模型完成random dots motion实验
B = 3.0
B_max = B
B_min = 2.0
drift_rate = 6
σ = 2
time_non_decision = 0.3
coherences = [-0.512, -.256, -.128, -.064, -.032, 0, +.032, +.064, +.128, +.256, +.512]
n_coherences = length(coherences)

total_trials = 10000

#simulating DDM

choices = []
rts = []
for coh in coherences
    push!(choices,[])
    push!(rts,[])
end

for trial in 1:total_trials
    accum = 0
    rt = 0
    coh_i = rand(1:n_coherences)
    coh = coherences[coh_i]
    
    rt, xs, decision = simulate_ddm(drift_rate * coh, σ, B, dt, max_t*2)
    if decision!=0 
        push!(choices[coh_i], decision/2+0.5)    # convert decision into 0 and 1
        push!(rts[coh_i], rt + time_non_decision)
    end
end

p_right=[]
rt=[]
for coh_i in 1:n_coherences
    if length(choices[coh_i])>0
        push!(p_right, mean(choices[coh_i]))
        push!(rt, mean(rts[coh_i]))
    else
        push!(p_right,-1)
        push!(rt,0)
    end
end

In [11]:
# collapsing bound

choices_collapse = []
rts_collapse = []
for coherence in coherences
    push!(choices_collapse, [])
    push!(rts_collapse, [])
end

for trial in 1:total_trials
    accum = 0
    rt = 0
    coh_index = rand(1:n_coherences)
    coherence = coherences[coh_index]
    
    rt, xs, decision, bs = simulate_ddm_with_collapsing_bound(drift_rate * coherence, σ, B_max, B_min, tau, dt, max_t * 2)
    if decision != 0 
        push!(choices_collapse[coh_index], decision / 2 + 0.5)
        push!(rts_collapse[coh_index], rt + time_non_decision)
    end
end

p_right_collapse = []
rt_collapse = []
for coh_index in 1:n_coherences
    if length(choices_collapse[coh_index]) > 0
        push!(p_right_collapse, mean(choices_collapse[coh_index]))
        push!(rt_collapse, mean(rts_collapse[coh_index]))
    else
        push!(p_right_collapse, -1)
        push!(rt_collapse, 0)
    end
end

p1 = plot(coherences, p_right_collapse, Layout(title="Psychometric Curve Collapsing", xaxis_title="Coherences", yaxis_title="Right Choices"))
p2 = plot(coherences, rt_collapse, Layout(title="Chronometric Curve Collapsing", xaxis_title="Coherences", yaxis_title="RT"))
p = [p1; p2]

┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields type, x, xaxis, y, and yaxis",
  "scatter with fields type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, margin, template, xaxis1, xaxis2, yaxis1, and yaxis2"

In [12]:
# Leaky 
choices_leaky = []
rts_leaky = []
for coh in coherences
    push!(choices_leaky, [])
    push!(rts_leaky, [])
end

for trial in 1:total_trials
    accum_leaky = 0
    rt_leaky = 0
    coh_i_leaky = rand(1:n_coherences)  
    coh_leaky = coherences[coh_i_leaky]


    rt_leaky, xs_leaky, decision_leaky = simulate_ddm_leaky_integration(drift_rate * coh_leaky, σ, B, λ, dt, max_t * 2)
    
    if decision_leaky != 0
        push!(choices_leaky[coh_i_leaky], decision_leaky / 2 + 0.5)  
        push!(rts_leaky[coh_i_leaky], rt_leaky + time_non_decision)
    end
end

p_right_leaky = []
rt_means_leaky = []
for coh_i in 1:n_coherences
    if length(choices_leaky[coh_i]) > 0
        push!(p_right_leaky, mean(choices_leaky[coh_i]))
        push!(rt_means_leaky, mean(rts_leaky[coh_i]))
    else
        push!(p_right_leaky, -1)
        push!(rt_means_leaky, 0)
    end
end


p1_leaky = plot(coherences, p_right_leaky, Layout(title="Psychometric Curve (Leaky)", xaxis_title="Coherences", yaxis_title="Right Choices"))


p2_leaky = plot(coherences, rt_means_leaky, Layout(title="Chronometric Curve (Leaky)", xaxis_title="Coherences", yaxis_title="Reaction Time (RT)"))


p_leaky = [p1_leaky; p2_leaky]
display(p_leaky)


┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that you will not be able to save figures using `savefig`.
│ 
│ If you are on Windows this might be caused by known problems with Kaleido v0.2 on Windows (you are using version 0.2.1).
│ You might want to try forcing a downgrade of the Kaleido_jll library to 0.1.
│ Check the Package Readme at https://github.com/JuliaPlots/PlotlyKaleido.jl/tree/main#windows-note for more details.
│ 
│ If you think this is not your case, you might try using a longer timeout to check if the process is not responding (defaults to 10 seconds) by passing the desired value in seconds using the `timeout` kwarg when calling `PlotlyKaleido.start` or `PlotlyKaleido.restart`
└ @ PlotlyKaleido C:\Users\86181\.julia\packages\PlotlyKaleido\U5CX4\src\PlotlyKaleido.jl:24
┌ Warning: It looks like the Kaleido process is not responding. 
│ The unresponsive process will be killed, but this means that y

data: [
  "scatter with fields type, x, xaxis, y, and yaxis",
  "scatter with fields type, x, xaxis, y, and yaxis"
]

layout: "layout with fields annotations, margin, template, xaxis1, xaxis2, yaxis1, and yaxis2"